In [1]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict

In [2]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float

    summary: str

In [7]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs']/state['balls'])*100
    return {'sr' : sr}

def calculate_bpb(state: BatsmanState):
    bpb = state['balls']/ (state['fours'] + state['sixes'])
    return {'bpb' : bpb}

def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6))/ state['runs']) * 100
    return {'boundary_percent' : boundary_percent}

def calculate_summary(state: BatsmanState):
    summary = f"""
strike rate - {state['sr']} \n balls per boundary - {state['bpb']} \n boundary percent - {state['boundary_percent']}
"""
    return {'summary' : summary}

In [9]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('calculate_summary', calculate_summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'calculate_summary')
graph.add_edge('calculate_bpb', 'calculate_summary')
graph.add_edge('calculate_boundary_percent', 'calculate_summary')

graph.add_edge('calculate_summary', END)

workflow = graph.compile()

In [12]:
intial_state = workflow.invoke({
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
})

In [13]:
intial_state

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percent': 48.0,
 'summary': '\nstrike rate - 200.0 \n balls per boundary - 5.0 \n boundary percent - 48.0\n'}